In [ ]:
import requests
import numpy as np

In [ ]:
filename = "/home/test_user1/test/streamtest.h5"  # the domain path we will use when querying the server
endpoint = "http://localhost:5101"     # the server address

In [ ]:
# get information about the domain
req = endpoint + "/"
headers = {"X-Hdf-domain": filename}
r = requests.get(req, headers=headers)
if r.status_code == 200:
    print(r.json())
elif r.status_code == 403:
    print("Access forbidden: run 'hsacl <path> +r default' to make public readable")
else:
    print(f"Error: {r.status_code} {r.reason}")

In [46]:
# get the names and ids of all sensors in the domain
sensor_ids = {}
req = endpoint + "/groups/"
headers = {"X-Hdf-domain": filename}
params = {"h5path": "/sensors", "include_links": 1}
r = requests.get(req, headers=headers, params=params)
if r.status_code == 200:
    rsp = r.json()
    if "hrefs" in rsp:
        del rsp["hrefs"]
    if "links" in rsp:
        links = rsp["links"]
        for title in links:
            link = links[title]
            sensor_ids[title] = link["id"]
else:
    print(f"Error: {r.status_code} {r.reason}")

print("sensor_name     sensor_id")
print("-----------     ---------")
for sensor_name in sensor_ids:
    sensor_id = sensor_ids[sensor_name]
    print(f"{sensor_name}\t\t{sensor_id}")


sensor_name     sensor_id
-----------     ---------
sensor1		d-71c30272-098758e0-d928-028a79-ef1534


In [47]:
# get inforamation about a specific sensor, will return shape, and attributes
print(f"getting attributes sensor: {sensor_name}")
req = endpoint + "/datasets/" + sensor_ids[sensor_name]
headers = {"X-Hdf-domain": filename}
params = {"include_attrs": 1}
r = requests.get(req, headers=headers, params=params)
if r.status_code == 200:
    rsp = r.json()
    attrs = rsp["attributes"]
    for attr_name in attrs:
        attr_json = attrs[attr_name]
        value = attr_json["value"]
        print(f"  {attr_name}: {value}")
else:
    print(f"Error: {r.status_code} {r.reason}")



getting attributes sensor: sensor1
  description: sensor data for sensor sensor1
  frequency: 100
  scale_factor: 100
  update_interval: 1.0
  start_time: 1763237041.1143281
  min_value: 0.0
  max_value: 200.0
  units: amps


In [48]:
# get min/max mean values for given sensor's data
req = endpoint + "/datasets/" + sensor_ids[sensor_name] + "/value"
headers = {"X-Hdf-domain": filename}

r = requests.get(req, headers=headers)
if r.status_code == 200:
    rsp = r.json()
    values = rsp["value"]
    arr = np.array(values)
    print(f"min: {arr.min():.2f}, max: {arr.max():.2f}, mean: {arr.mean():.2f}")
    
else:
    print(f"Error: {r.status_code} {r.reason}")

min: 31.65, max: 183.21, mean: 94.38


In [ ]:
# get sensor data values in the range start to stop
start = 10
stop = 20
req = endpoint + "/datasets/" + sensor_ids[sensor_name] + "/value"
headers = {"X-Hdf-domain": filename}

params = {"select": f"[{start}:{stop}]"}
r = requests.get(req, headers=headers, params=params)
if r.status_code == 200:
    rsp = r.json()
    values = rsp["value"]
    index = start
    for value in values:
        print(f"value[{index}]: {value:.2f}")
        index += 1
else:
    print(f"Error: {r.status_code} {r.reason}")